In [27]:
import newton
import numpy as np
import warp as wp

newton.use_coord_layout_targets = True

In [20]:
# Helper functions

import json
import os
import time
import uuid
from html import escape
from pathlib import Path

from IPython.display import HTML, Javascript, display

# Optional: force CPU for debugging (default is GPU if available)
use_cpu = False
if use_cpu:
    wp.set_device("cpu")


def make_viewer(name: str, fixed_dpr: float = 2.0):
    """Create a replayable Viser recording for an output cell.

    ``fixed_dpr`` pins the playback iframe to a fixed devicePixelRatio (via the
    ``?fixedDpr=`` URL param the viser client reads from ``DevSettingsStore``).
    Patching the shared ``_camera_query_from_request`` staticmethod lets the
    param ride through both the live-server iframe URL and the Sphinx static
    embed.
    """
    recording_path = Path("../_static/recordings") / f"{name}.viser"
    recording_path.parent.mkdir(parents=True, exist_ok=True)
    viewer = newton.viewer.ViewerViser(verbose=False, record_to_viser=str(recording_path))

    cls = newton.viewer.ViewerViser
    if not hasattr(cls, "_orig_camera_query_from_request"):
        cls._orig_camera_query_from_request = cls._camera_query_from_request

    def _patched(camera_request):
        return cls._orig_camera_query_from_request(camera_request) + f"&fixedDpr={fixed_dpr}"

    cls._camera_query_from_request = staticmethod(_patched)

    return viewer


def render_mermaid(diagram: str, theme: str = "forest", line_color: str = "#76b900", width: str = "100%"):
    """Render Mermaid text as a diagram in Jupyter output."""
    element_id = f"mermaid-{uuid.uuid4().hex}"
    diagram_html = escape(diagram)
    width_style = escape(width, quote=True)
    display(HTML(f'<pre id="{element_id}" class="mermaid" style="width: {width_style};">{diagram_html}</pre>'))

    if os.environ.get("NEWTON_SPHINX_BUILD") == "1":
        # sphinxcontrib-mermaid renders pre.mermaid blocks in the built docs.
        return

    config_json = json.dumps(
        {
            "startOnLoad": False,
            "theme": theme,
            "themeVariables": {"lineColor": line_color},
        }
    )

    js = f"""
(async () => {{
  const loadMermaid = () => new Promise((resolve, reject) => {{
    if (window.mermaid) {{
      resolve();
      return;
    }}
    const script = document.createElement("script");
    script.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    script.onload = () => resolve();
    script.onerror = () => reject(new Error("Failed to load Mermaid."));
    document.head.appendChild(script);
  }});

  await loadMermaid();
  const config = {config_json};
  window.mermaid.initialize(config);

  const node = document.getElementById("{element_id}");
  if (!node) {{
    return;
  }}

  try {{
    await window.mermaid.run({{ nodes: [node] }});
  }} catch (err) {{
    node.textContent = "Mermaid render error: " + (err && err.message ? err.message : err);
  }}
}})();
"""
    display(Javascript(js))


class _HTMLProgressBar:
    """Frontend-agnostic HTML progress bar for notebook loops."""

    def __init__(self, iterable, total: int, desc: str = "", leave: bool = True):
        self._iterable = iterable
        self.total = max(int(total), 1)
        self.desc = desc
        self.leave = leave
        self.n = 0
        self._start = time.perf_counter()
        self._last_refresh = 0.0
        self._handle = display(HTML(self._render()), display_id=True)

    def _render(self) -> str:
        elapsed = max(time.perf_counter() - self._start, 1e-9)
        pct = min(100.0, 100.0 * self.n / self.total)
        rate = self.n / elapsed
        remaining = (self.total - self.n) / rate if rate > 1e-9 else float("inf")
        eta = "--:--" if not np.isfinite(remaining) else f"{int(remaining // 60):02d}:{int(remaining % 60):02d}"
        desc_html = escape(self.desc)
        return (
            f'<div style="font-family: sans-serif; margin: 6px 0;">'
            f'<div style="display:flex; justify-content:space-between; font-size:12px; margin-bottom:4px;">'
            f"<span>{desc_html}</span>"
            f"<span>{self.n}/{self.total} ({pct:5.1f}%)</span>"
            f"</div>"
            f'<progress value="{self.n}" max="{self.total}" style="width:100%; height:14px;"></progress>'
            f'<div style="font-size:11px; color:#666; margin-top:2px;">elapsed {elapsed:5.1f}s | eta {eta}</div>'
            f"</div>"
        )

    def _refresh(self, force: bool = False):
        now = time.perf_counter()
        if force or (now - self._last_refresh) >= 0.1 or self.n >= self.total:
            self._handle.update(HTML(self._render()))
            self._last_refresh = now

    def set_description(self, desc: str):
        self.desc = desc
        self._refresh(force=True)

    def set_description_str(self, desc: str):
        self.set_description(desc)

    def update(self, n: int = 1):
        self.n = min(self.total, self.n + int(n))
        self._refresh()

    def close(self):
        if self.leave:
            self._refresh(force=True)
        else:
            self._handle.update(HTML(""))

    def __iter__(self):
        try:
            for item in self._iterable:
                yield item
                self.update(1)
        finally:
            self.close()


def _tqdm_html(iterable=None, total=None, desc: str = "", leave: bool = False, **_kwargs):
    if iterable is None:
        if total is None:
            raise ValueError("Either iterable or total must be provided.")
        iterable = range(int(total))
    if total is None:
        try:
            total = len(iterable)
        except TypeError as e:
            raise ValueError("total is required for iterables without len().") from e
    return _HTMLProgressBar(iterable=iterable, total=int(total), desc=desc, leave=leave)


def _trange_html(*args, **kwargs):
    return _tqdm_html(range(*args), **kwargs)


# Use HTML progress bars for reliable updates in this frontend.
tqdm, trange = _tqdm_html, _trange_html


AXIS_COLORS = wp.array([(1.0, 0.1, 0.1), (0.1, 1.0, 0.1), (0.1, 0.4, 1.0)], dtype=wp.vec3)
AXIS_HALF = 0.05
_AXIS_BASIS = (wp.vec3(1.0, 0.0, 0.0), wp.vec3(0.0, 1.0, 0.0), wp.vec3(0.0, 0.0, 1.0))


def log_frame_axes(viewer, name, pos, quat=None, *, half=AXIS_HALF, width=0.01):
    """Log a 3-line RGB axis indicator at ``pos`` rotated by xyzw ``quat``.

    ``pos`` may be a 3-vec or a 7-element ``wp.transform``; in the latter case
    the rotation is read from the transform and ``quat`` is ignored.
    """
    arr = np.asarray(pos, dtype=np.float32).reshape(-1)
    if arr.size == 7:
        position, rotation = arr[:3], arr[3:7]
    elif arr.size == 3:
        position = arr
        rotation = quat if quat is not None else (0.0, 0.0, 0.0, 1.0)
    else:
        raise ValueError(f"log_frame_axes: pos must be length 3 or 7, got {arr.size}")
    q = wp.quat(*(float(c) for c in rotation))
    starts = np.tile(position, (3, 1))
    ends = np.stack([position + half * np.asarray(wp.quat_rotate(q, b), dtype=np.float32) for b in _AXIS_BASIS])
    viewer.log_lines(name, starts, ends, colors=AXIS_COLORS, width=width)

In [21]:
FRANKA_FINGER_CLOSED = 0.0
FRANKA_FINGER_OPEN = 0.04

# Franka initial joint positions (7 arm joints [rad] + 2 gripper fingers [m])
FRANKA_HOME_Q = [
    0.0,
    0.02,
    0.0,
    -2.37,
    0.0,
    2.39,
    np.pi / 4,
    FRANKA_FINGER_OPEN,
    FRANKA_FINGER_OPEN,
]

FRANKA_DOF_COUNT = 9
FRANKA_ARM_DOF_COUNT = 7

# These values set controller limits and solver regularization for the first 9 Franka DOFs.
FRANKA_EFFORT_LIMITS = [87, 87, 87, 87, 12, 12, 12, 100, 100]
FRANKA_MUJOCO_ARMATURE = [0.195] * 4 + [0.074] * 3 + [0.1] * 2

In [22]:
def build_franka_scene(include_table=True, include_cube=True):
    """Build a Franka + table scene and optionally add a cube.

    Args:
        include_table: Whether to add a table.
        include_cube: Whether to add a cube on the table.

    Returns:
        Tuple of (builder, cube_size, table_pos, cube_body).
    """
    builder = newton.ModelBuilder()
    builder.default_shape_cfg.gap = 0.0
    newton.solvers.SolverMuJoCo.register_custom_attributes(builder)

    table_height = 0.1
    table_pos = wp.vec3(0.0, -0.5, 0.5 * table_height)
    table_top_center = table_pos + wp.vec3(0.0, 0.0, 0.5 * table_height)
    if include_table:
        builder.add_shape_box(
            body=-1,
            hx=0.4,
            hy=0.4,
            hz=0.5 * table_height,
            xform=wp.transform(table_pos),
        )

    robot_base_pos = table_top_center + wp.vec3(-0.5, 0.0, 0.0)
    builder.add_urdf(
        str("../assets/franka_emika_panda/urdf/fr3_franka_hand.urdf"),
        xform=wp.transform(robot_base_pos, wp.quat_identity()),
        floating=False,
        enable_self_collisions=False,
        parse_visuals_as_colliders=False,
    )

    builder.joint_q[:FRANKA_DOF_COUNT] = FRANKA_HOME_Q
    builder.joint_effort_limit[:FRANKA_DOF_COUNT] = FRANKA_EFFORT_LIMITS
    builder.joint_armature[:FRANKA_DOF_COUNT] = FRANKA_MUJOCO_ARMATURE

    cube_size = 0.05
    cube_body = None
    if include_cube:
        cube_pos = table_top_center + wp.vec3(0.0, 0.15, 0.5 * cube_size)
        cube_body = builder.add_body(xform=wp.transform(cube_pos, wp.quat_identity()))
        shape_cfg = newton.ModelBuilder.ShapeConfig(margin=1e-3, density=400.0)
        builder.add_shape_box(
            body=cube_body,
            hx=0.5 * cube_size,
            hy=0.5 * cube_size,
            hz=0.5 * cube_size,
            cfg=shape_cfg,
        )

    return builder, cube_size, table_pos, cube_body

In [23]:
# Build a scene without a cube; this section drives one joint with raw force.
builder, _, _, _ = build_franka_scene(include_cube=False)

model = builder.finalize()

state_0 = model.state()
state_1 = model.state()
control = model.control()
collision_pipeline = newton.CollisionPipeline(model)
contacts = collision_pipeline.contacts()

# Forward kinematics: compute body transforms from the initial joint_q.
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

viewer = make_viewer("02_franka_scene")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)
viewer.begin_frame(0)
viewer.log_state(state_0)
viewer.end_frame()
viewer

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

Module newton._src.viewer.kernels b978a3c load on device 'cpu' took 1.65 ms  (cached)


In [40]:
FRANKA_TARGET_KE = [900, 900, 700, 700, 400, 400, 400, 100, 100]
FRANKA_TARGET_KD = [90, 90, 70, 70, 40, 40, 40, 10, 10]
FRANKA_BODY_GRAVCOMP = 1.0


def enable_franka_target_control(builder):
    """Enable position targets and MuJoCo gravity compensation for Franka."""
    builder.joint_target_q[:FRANKA_DOF_COUNT] = FRANKA_HOME_Q
    builder.joint_target_ke[:FRANKA_DOF_COUNT] = FRANKA_TARGET_KE
    builder.joint_target_kd[:FRANKA_DOF_COUNT] = FRANKA_TARGET_KD

    joint_actgravcomp = builder.custom_attributes["mujoco:jnt_actgravcomp"].values
    for dof_index in range(FRANKA_DOF_COUNT):
        joint_actgravcomp[dof_index] = True

    body_gravcomp = builder.custom_attributes["mujoco:gravcomp"].values
    for body_index, label in enumerate(builder.body_label):
        # Skip the world-fixed base bodies; compensate the moving arm, hand, and fingers.
        if label.startswith("fr3/") and label not in {"fr3/base", "fr3/fr3_link0"}:
            body_gravcomp[body_index] = FRANKA_BODY_GRAVCOMP

In [41]:
builder, cube_size, table_pos, cube_body = build_franka_scene(include_cube=True)
enable_franka_target_control(builder)
model = builder.finalize()

In [37]:
state_0 = model.state()
state_1 = model.state()
control = model.control()
collision_pipeline = newton.CollisionPipeline(model)
contacts = collision_pipeline.contacts()

solver = newton.solvers.SolverMuJoCo(
    model,
    solver="newton",
    integrator="implicitfast",
    iterations=20,
    ls_iterations=100,
    nconmax=1000,
    njmax=2000,
    cone="elliptic",
    impratio=1000.0,
    use_mujoco_contacts=False,
)

newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)

In [38]:
fps = 10
frame_dt = 1.0 / fps
sim_substeps = 100
sim_dt = frame_dt / sim_substeps

In [39]:
graph = None

def run_sim_substeps():
    global state_0, state_1
    for _ in range(sim_substeps):
        state_0.clear_forces()
        collision_pipeline.collide(state_0, contacts)
        solver.step(state_in=state_0, state_out=state_1, control=control, contacts=contacts, dt=sim_dt)
        state_0, state_1 = state_1, state_0

In [43]:
viewer = make_viewer("04_franka_joint_targets")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.5, 0.0, 0.5), -15, -140)

╭────── viser (listening *:8081) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8081   │
│   Websocket │ ws://localhost:8081     │
│             ╵                         │
╰───────────────────────────────────────╯

In [44]:
total_frames = 50
sim_time = 0.0

for frame in range(total_frames):
    target = model.joint_q.numpy().astype(np.float32)
    control.joint_target_q.assign(target)

    if graph is not None:
        wp.capture_launch(graph)
    else:
        with  wp.ScopedCapture() as capture:
            run_sim_substeps()
        graph = capture.graph

    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.end_frame()
    sim_time += frame_dt

viewer